## Data Integration — Combining

In [1]:
import pandas as pd
import glob

# Automatically pick up all 12 files (sorted: 1, 2, ... 12)
files = sorted(glob.glob("../data/processed/df_cleaned_*.csv"),
               key=lambda f: int(f.split("_")[-1].split(".")[0]))

dfs = [pd.read_csv(f) for f in files]

# Check schemas match before merging
for f, d in zip(files, dfs):
    print(f, d.shape)

# Integrate (stack rows on top of each other)
df_integrated = pd.concat(dfs, ignore_index=True)



../data/processed/df_cleaned_1.csv (3561388, 23)
../data/processed/df_cleaned_2.csv (3250954, 23)
../data/processed/df_cleaned_3.csv (3811060, 23)
../data/processed/df_cleaned_4.csv (3810503, 23)
../data/processed/df_cleaned_5.csv (4347294, 23)
../data/processed/df_cleaned_6.csv (4117575, 23)
../data/processed/df_cleaned_7.csv (3702899, 23)
../data/processed/df_cleaned_8.csv (3385830, 23)
../data/processed/df_cleaned_9.csv (4052538, 23)
../data/processed/df_cleaned_10.csv (4202857, 23)
../data/processed/df_cleaned_11.csv (3952579, 23)
../data/processed/df_cleaned_12.csv (4106224, 23)


The cleaned data is split across 12 monthly files. Each was cleaned independently using
the same pipeline, so the schemas are identical and the files can be stacked row-wise
into a single dataset for modelling.

Files are sorted **numerically** rather than alphabetically — a plain sort would order
them `1, 10, 11, 12, 2, 3, ...`, scrambling the chronological sequence that the
time-based train/validation/test split depends on later.

Shapes are printed per file before concatenation so that any month with an unexpected
row or column count is visible before it is silently absorbed into the combined frame.

In [4]:
zones = pd.read_csv("../data/raw/Urban_Flow_Analytics_Zone_Dataset.csv")


# Pickup zone info
df_integrated = df_integrated.merge(
    zones.rename(columns={
        'loc_id': 'origin_loc_id',
        'borough_name': 'pickup_borough',
        'zone_name': 'pickup_zone',
        'service_zone': 'pickup_service_zone'
    }),
    on='origin_loc_id', how='left'
)

# Dropoff zone info
df_integrated = df_integrated.merge(
    zones.rename(columns={
        'loc_id': 'dest_loc_id',
        'borough_name': 'dropoff_borough',
        'zone_name': 'dropoff_zone',
        'service_zone': 'dropoff_service_zone'
    }),
    on='dest_loc_id', how='left'
)

print("Shape after joins:", df_integrated.shape)
df_integrated.head()

Shape after joins: (46301701, 29)


,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,...,congestion_relief_fee,rider_count_missing,calculated_charge,rider_count_was_zero,pickup_borough,pickup_zone,pickup_service_zone,dropoff_borough,dropoff_zone,dropoff_service_zone
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,0.00,False,15.86,False,Manhattan,Upper West Side South,Yellow Zone,Manhattan,Upper West Side North,Yellow Zone
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,1.0,0.90,1.0,N,163,162,2,...,0.75,False,16.90,True,Manhattan,Midtown North,Yellow Zone,Manhattan,Midtown East,Yellow Zone
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,1.0,1.40,1.0,N,43,237,1,...,0.75,False,22.20,True,Manhattan,Central Park,Yellow Zone,Manhattan,Upper East Side South,Yellow Zone
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,0.75,False,55.56,False,Manhattan,Lincoln Square East,Yellow Zone,Manhattan,Seaport,Yellow Zone
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,1.0,2.16,1.0,N,88,144,1,...,0.75,False,23.10,True,Manhattan,Financial District South,Yellow Zone,Manhattan,Little Italy/NoLiTa,Yellow Zone


In [5]:
print(f"integrated: {len(df_integrated):,} rows from {len(files)} files")
df_integrated.to_csv("../data/processed/df_integrated.csv", index=False)

integrated: 46,301,701 rows from 12 files
